In [5]:
# ============================================================
# FULL PIPELINE FOR LOBSTER BIOACOUSTICS CLASSIFICATION
# VERSION: HARDWARE-AGNOSTIC STABLE (FORCE CPU FOR ALL)
# ============================================================

import os
import sys

# CRITICAL: These must be set BEFORE any other imports to ensure 
# that no library (XGBoost or TensorFlow) attempts to touch the GPU.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import librosa
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

# Classical Models
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Deep Learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense

# Confirming CPU usage
print(f"TensorFlow is using: {'CPU' if len(tf.config.list_physical_devices('GPU')) == 0 else 'GPU (Warning)'}")

# ============================================================
# DATASET ROOT
# ============================================================
BASE_DATASET_DIR = "/home/feliciano/LOBSTER SOUNDS/DatasetClass"

# ============================================================
# REFINED FEATURE EXTRACTION
# ============================================================

def extract_refined_features(audio, sr):
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    mfcc_mean = np.mean(mfcc, axis=1)
    
    centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)
    centroid_mean = np.mean(centroid, axis=1)
    
    contrast = librosa.feature.spectral_contrast(y=audio, sr=sr)
    contrast_mean = np.mean(contrast, axis=1)
    
    rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr)
    rolloff_mean = np.mean(rolloff, axis=1)
    
    zcr = librosa.feature.zero_crossing_rate(y=audio)
    zcr_mean = np.mean(zcr, axis=1)
    
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)
    
    return np.concatenate([mfcc_mean, centroid_mean, contrast_mean, rolloff_mean, zcr_mean, chroma_mean])

def load_audio_folder(folder, label):
    X, y = [], []
    if not os.path.exists(folder):
        print(f"Missing: {folder}")
        return np.array([]), np.array([])
    for f in os.listdir(folder):
        if f.lower().endswith(".wav"):
            try:
                audio, sr = librosa.load(os.path.join(folder, f), sr=22050)
                if len(audio) == 0: continue
                X.append(extract_refined_features(audio, sr))
                y.append(label)
            except:
                continue
    return np.array(X), np.array(y)

# Load Data
print("Step 1: Extracting Fusion Features (CPU)...")
X_f, y_f = load_audio_folder(os.path.join(BASE_DATASET_DIR, "juvenile_lobsters"), 1)
X_m, y_m = load_audio_folder(os.path.join(BASE_DATASET_DIR, "adult_lobsters"), 0)

X = np.vstack([X_f, X_m])
y = np.concatenate([y_f, y_m])

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ============================================================
# BOOTSTRAP METRICS
# ============================================================

def bootstrap_ci(y_true, y_pred_or_prob, metric, n=1000):
    rng = np.random.default_rng(42)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2: continue
        vals.append(metric(y_true[idx], y_pred_or_prob[idx]))
    return (np.percentile(vals, 2.5), np.percentile(vals, 97.5)) if vals else (0.0, 0.0)

# ============================================================
# EXECUTION LOOP
# ============================================================

results = []

# Classical Models
models = {
    "Random Forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(n_jobs=-1, random_state=42), 
    "MLP": MLPClassifier(hidden_layer_sizes=(128,), max_iter=1000, random_state=42),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(probability=True, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
}

print("Step 2: Training Classical Models...")
for name, model in models.items():
    model.fit(X_train, y_train)
    yp = model.predict(X_test)
    yp_p = model.predict_proba(X_test)[:, 1]
    
    acc_ci = bootstrap_ci(y_test, yp, accuracy_score)
    auc_ci = bootstrap_ci(y_test, yp_p, roc_auc_score)
    
    results.append([name, f"{accuracy_score(y_test, yp)*100:.2f}", 
                    f"{acc_ci[0]*100:.2f}-{acc_ci[1]*100:.2f}", 
                    f"{roc_auc_score(y_test, yp_p)*100:.2f}", 
                    f"{auc_ci[0]*100:.2f}-{auc_ci[1]*100:.2f}"])

# Deep Learning (Forcing CPU context for Keras)
X_train_dl = X_train[..., np.newaxis]
X_test_dl = X_test[..., np.newaxis]

print("Step 3: Training Deep Learning Models (Forced CPU)...")
with tf.device('/CPU:0'):
    for depth in [1, 2, 3, 4]:
        for dilated in [False, True]:
            name = f"1D-{'D' if dilated else ''}CNN ({depth}L)"
            
            m = Sequential([Input(shape=(X_train.shape[1], 1))])
            for i in range(depth):
                m.add(Conv1D(64, 3, dilation_rate=2 if dilated else 1, padding="same", activation="relu"))
                if i < depth - 1: m.add(MaxPooling1D(2))
            m.add(Flatten())
            m.add(Dense(128, activation="relu"))
            m.add(Dense(1, activation="sigmoid"))
            m.compile(optimizer="adam", loss="binary_crossentropy")
            
            m.fit(X_train_dl, y_train, epochs=10, batch_size=32, verbose=0)
            yp_p = m.predict(X_test_dl, verbose=0).ravel()
            yp = (yp_p > 0.5).astype(int)
            
            acc_ci = bootstrap_ci(y_test, yp, accuracy_score)
            auc_ci = bootstrap_ci(y_test, yp_p, roc_auc_score)
            
            results.append([name, f"{accuracy_score(y_test, yp)*100:.2f}", 
                            f"{acc_ci[0]*100:.2f}-{acc_ci[1]*100:.2f}", 
                            f"{roc_auc_score(y_test, yp_p)*100:.2f}", 
                            f"{auc_ci[0]*100:.2f}-{auc_ci[1]*100:.2f}"])

# Final Report
df = pd.DataFrame(results, columns=["Model", "Acc (%)", "95% CI", "AUC (%)", "95% CI"])
print("\n" + "="*80)
print(df.to_string(index=False))

TensorFlow is using: GPU (Warning)
Step 1: Extracting Fusion Features (CPU)...
Step 2: Training Classical Models...
Step 3: Training Deep Learning Models (Forced CPU)...

        Model Acc (%)      95% CI AUC (%)      95% CI
Random Forest   97.40 96.65-98.15   99.68 99.50-99.83
      XGBoost   97.81 97.06-98.50   99.78 99.66-99.88
          MLP   98.43 97.74-99.04   99.89 99.81-99.95
  Naive Bayes   91.04 89.60-92.48   97.66 96.98-98.28
          SVM   98.43 97.81-98.97   99.88 99.79-99.95
          KNN   98.56 97.95-99.11   99.43 98.98-99.81
  1D-CNN (1L)   98.08 97.40-98.77   99.89 99.82-99.95
 1D-DCNN (1L)   98.29 97.61-98.97   99.89 99.81-99.95
  1D-CNN (2L)   97.67 96.85-98.43   99.81 99.69-99.91
 1D-DCNN (2L)   97.81 97.06-98.50   99.78 99.63-99.91
  1D-CNN (3L)   97.81 97.06-98.50   99.79 99.66-99.89
 1D-DCNN (3L)   97.20 96.30-98.02   99.62 99.40-99.80
  1D-CNN (4L)   98.15 97.54-98.77   99.81 99.70-99.91
 1D-DCNN (4L)   97.40 96.51-98.22   99.56 99.24-99.80
